# Claude Managed Agents

## A Simple Demo of a Research Agent in 5 steps

We define the environment - Claude Managed Agents runs the loop in the cloud

To setup:

1. [Install uv](https://docs.astral.sh/uv/getting-started/installation/) if not already installed
2. Open a terminal and run `uv sync`
3. Create a file called `.env` and add one line: `ANTHROPIC_API_KEY=sk-ant-xxx`

In [ ]:
from anthropic import Anthropic
from dotenv import load_dotenv
load_dotenv(override=True)

## Step 1: Create the Agent

In [ ]:
client = Anthropic()

agent = client.beta.agents.create(
    name="Researcher",
    model="claude-sonnet-4-6",
    system="You are able to carry out research tasks using tools at your disposal.",
    tools=[
        {"type": "agent_toolset_20260401"},
    ],
)

print(f"Agent ID: {agent.id}, version: {agent.version}")

## Step 2: Create the Environment

In [ ]:
environment = client.beta.environments.create(
    name="my-own-environment",
    config={
        "type": "cloud",
        "networking": {"type": "unrestricted"},
    },
)

print(f"Environment ID: {environment.id}")

## Step 3: Create a Session

In [ ]:
session = client.beta.sessions.create(
    agent=agent.id,
    environment_id=environment.id,
    title="Quickstart session",
)

print(f"Session ID: {session.id}")

## Step 4: Create an Event

In [ ]:
prompt = """
Please research a high quality 60 inch smart LED 4K TV that gets great feedback from customers,
and has good contrast and picture quality, and is reasonably priced. Give your top 3 recommendations with the best onine
price and a link to where I can buy each.
"""

In [ ]:
events = [{"type": "user.message", "content": [{"type": "text", "text": prompt}]}]

## Step 5: Send the Event and stream back Events

In [ ]:
with client.beta.sessions.events.stream(session.id) as stream:
    client.beta.sessions.events.send(session.id, events=events)

    for event in stream:
        match event.type:
            case "agent.message":
                for block in event.content:
                    print(block.text, end="")
            case "agent.tool_use":
                print(f"\n[Using tool: {event.name}]")
            case "session.status_idle":
                print("\n\nAgent finished.")
                break


## And now run this as a gradio app

In a terminal:

```bash
uv run app.py
```